In [2]:
import os
import numpy as np

import qkeras
from qkeras import QActivation
from qkeras import QConv1D
from qkeras.quantizers import quantized_bits, quantized_relu, quantized_tanh

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger



filters = 64
name =f"no_opt_{filters}_regular"
X_total = np.load(f"../Data/X_Data_Bank_Final.npy")
Y_total = np.load(f"../Data/Y_Data_Bank_Final.npy")
X_data = X_total[:1500,:,:]
y_data = Y_total[:1500,:,:]
TIME_STEPS = np.shape(X_data)[1]

VAL_SPLIT = 0.1

SAVE_DIR_opt = f'../Extra_Files/Output_drives/1Conv_checkpoints_running'
LOG_FILE_opt = f'../Extra_Files/Log_files/1Conv_3pulse_noise_tb23_{name}.csv'
MODEL_NAME_TEMPLATE_opt = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv_' + f'{name}.h5'
checkpoint_path_opt = os.path.join(SAVE_DIR_opt, MODEL_NAME_TEMPLATE_opt)

model = Sequential([
    Input(shape=(TIME_STEPS, 1)),
    Conv1D(filters, 3, padding='same', activation = "relu"),
    Conv1D(filters, 3, padding='same'),
    Conv1D(filters, 3, padding='same'),
    Conv1D(1, 1, padding='same'),
    Activation('sigmoid')
    ])

model.summary()


optimizer = Adam(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer) 


if not os.path.isdir(SAVE_DIR_opt):
    os.makedirs(SAVE_DIR_opt)

callbacks = [
    ModelCheckpoint(checkpoint_path_opt, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
    CSVLogger(LOG_FILE_opt, append=True, separator=','),
    EarlyStopping(monitor="val_loss", mode="min", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1) 
]


model.fit(
    X_data, y_data,
    epochs=300,                  
    shuffle=True,
    validation_split=VAL_SPLIT,
    callbacks=callbacks
)
    
model.save(f"../Models/Fast_{filters}.h5")

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_4 (Conv1D)           (None, 80, 64)            256       
                                                                 
 conv1d_5 (Conv1D)           (None, 80, 64)            12352     
                                                                 
 conv1d_6 (Conv1D)           (None, 80, 64)            12352     
                                                                 
 conv1d_7 (Conv1D)           (None, 80, 1)             65        
                                                                 
 activation_1 (Activation)   (None, 80, 1)             0         
                                                                 
Total params: 25,025
Trainable params: 25,025
Non-trainable params: 0
_________________________________________________________________
Epoch 1/300
42/43 [============================>.]

Epoch 24/300
24/43 [===============>..............] - ETA: 0s - loss: 0.1133
Epoch 24: val_loss did not improve from 0.11412
43/43 [==============================] - 0s 3ms/step - loss: 0.1133 - val_loss: 0.1144 - lr: 0.0010
Epoch 25/300
23/43 [===============>..............] - ETA: 0s - loss: 0.1144
Epoch 25: val_loss improved from 0.11412 to 0.11405, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.11405.e025_deconv_no_opt_64_regular.h5

Epoch 25: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
43/43 [==============================] - 0s 3ms/step - loss: 0.1141 - val_loss: 0.1141 - lr: 0.0010
Epoch 26/300
22/43 [==============>...............] - ETA: 0s - loss: 0.1126
Epoch 26: val_loss improved from 0.11405 to 0.11371, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.11371.e026_deconv_no_opt_64_regular.h5
43/43 [==============================] - 0s 3ms/step - loss: 0.11

In [4]:
import hls4ml
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)
print(co)
hls_config = hls4ml.utils.config_from_keras_model(
    model,
    granularity='name',
    backend='Vitis'
)
hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=hls_config,
    backend='Vitis',
    output_dir='../HLS_models/model_final',
    part='xcu250-figd2104-2L-e',
    io_type='io_stream',
)
hls_model.compile()

{'QInitializer': <class 'qkeras.qlayers.QInitializer'>, 'QDense': <class 'qkeras.qlayers.QDense'>, 'QConv1D': <class 'qkeras.qconvolutional.QConv1D'>, 'QConv2D': <class 'qkeras.qconvolutional.QConv2D'>, 'QConv2DTranspose': <class 'qkeras.qconvolutional.QConv2DTranspose'>, 'QSimpleRNNCell': <class 'qkeras.qrecurrent.QSimpleRNNCell'>, 'QSimpleRNN': <class 'qkeras.qrecurrent.QSimpleRNN'>, 'QLSTMCell': <class 'qkeras.qrecurrent.QLSTMCell'>, 'QLSTM': <class 'qkeras.qrecurrent.QLSTM'>, 'QGRUCell': <class 'qkeras.qrecurrent.QGRUCell'>, 'QGRU': <class 'qkeras.qrecurrent.QGRU'>, 'QBidirectional': <class 'qkeras.qrecurrent.QBidirectional'>, 'QDepthwiseConv2D': <class 'qkeras.qconvolutional.QDepthwiseConv2D'>, 'QSeparableConv1D': <class 'qkeras.qconvolutional.QSeparableConv1D'>, 'QSeparableConv2D': <class 'qkeras.qconvolutional.QSeparableConv2D'>, 'QActivation': <class 'qkeras.qlayers.QActivation'>, 'QAdaptiveActivation': <class 'qkeras.qlayers.QAdaptiveActivation'>, 'QBatchNormalization': <class